In [7]:
""" Module for extracting network/pathway data using rpy2 package """

# import required package
import pandas as pd
import numpy as np
import json

import rpy2.robjects.packages as rpackages
# from rpy2.robjects import pandas2ri
# from rpy2.robjects.conversion import localconverter
import rpy2.robjects as robj


# path to datafiles directory
DATAFILES = 'datafiles/'


In [8]:

# Curated Melanoma Network data from the Virtual Cell

def extract_melanoma_vcells(path=DATAFILES+'MelanomaMap_vcells.cyjs'):
    ''' Extract Melanoma network from Virtual cell. '''
    # Load data
    network_data = json.load(open(path))
    # select nodes which are either protein or gene
    nodes = pd.DataFrame(pd.DataFrame(network_data['elements']['nodes'])['data'].to_list())
    nodes = nodes[nodes['class'].isin(['GENE', 'PROTEIN'])][['id', 'hgnc_symbol']]
    nodes.dropna(inplace=True)
    # get edge data for protein and genes
    edges = pd.DataFrame(pd.DataFrame(network_data['elements']['edges'])['data'].to_list())
    edges = edges[['source', 'target']]
    edges = edges[(edges.target.isin(nodes.id.unique())) & (edges.source.isin(nodes.id.unique()))]
    # map hgnc symbols to edge data
    network = pd.DataFrame(columns=['node1', 'node2'])
    network['node1'] = edges['source'].apply(lambda x: nodes[nodes.id == x]['hgnc_symbol'].values[0])
    network['node2'] = edges['target'].apply(lambda x: nodes[nodes.id == x]['hgnc_symbol'].values[0])
    network = network[network.node1 != network.node2].drop_duplicates()
    # processed melanoma network
    # network_rev = pd.DataFrame({'node1' : network['node2'], 'node2': network['node1']})
    # network_processed = pd.concat([network, network_rev], ignore_index=True).drop_duplicates()
    network.reset_index(drop=True, inplace=True)
    
    return network


In [10]:
# extract processed Melanoma network from Virtual cell
melanoma_vcells = extract_melanoma_vcells()
display(melanoma_vcells)
# save extracted data
melanoma_vcells.to_csv(DATAFILES+'Melanoma_vcells.csv', index=False)

,node1,node2
0,MAP3K3,MAPK14
1,CASP8,CASP3
2,CASP8,BID
3,MAP3K1,MAP2K7
4,MAP3K1,MAP2K4
...,...,...
130,PPP3CA,NFATC3
131,CASP3,BAD
132,SKI,SMAD2
133,CASP9,CASP7
